# 02. Treinamento e Fine-Tuning do Modelo YOLO para Placas Veiculares

Este notebook foi projetado para execução no **Google Colab com GPU**.
Ele realiza o fine-tuning do detector YOLOv8n em dataset de placas veiculares e exporta o modelo treinado em formato **ONNX** para uso no aplicativo desktop offline.

### 1. Configuração do Ambiente e Verificação de GPU

In [ ]:
!pip install -q ultralytics onnx onnxruntime

import torch
print('PyTorch Version:', torch.__version__)
print('GPU Disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
else:
    print('ATENÇÃO: Ative a GPU em Ambiente de Execução -> Alterar tipo de ambiente de execução.')

### 2. Download e Estruturação do Dataset
Utilizaremos a base *keremberke/license-plate-object-detection* ou imagens anotadas em formato YOLO.

In [ ]:
# Exemplo com dataset Roboflow / Hugging Face ou imagens próprias no Google Drive
import os

# Criação do arquivo de configuração placas.yaml
config_yaml = '''
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: placa
'''

with open('placas.yaml', 'w', encoding='utf-8') as f:
    f.write(config_yaml.strip())

print('placas.yaml configurado com sucesso!')

### 3. Fine-Tuning do Modelo YOLOv8n

In [ ]:
from ultralytics import YOLO

# Carrega modelo base pré-treinado leve
model = YOLO('yolov8n.pt')

# Executa treinamento
resultados = model.train(
    data='placas.yaml',
    epochs=50,
    imgsz=640,
    patience=15,
    batch=16,
    project='runs/detect',
    name='treino_placas'
)

### 4. Avaliação das Métricas no Conjunto de Teste

In [ ]:
# Avalia o melhor checkpoint
melhor_modelo = YOLO('runs/detect/treino_placas/weights/best.pt')
metricas = melhor_modelo.val(data='placas.yaml', split='test')

print('mAP@0.5:', metricas.box.map50)
print('mAP@0.5:0.95:', metricas.box.map)
print('Precisão:', metricas.box.p)
print('Recall:', metricas.box.r)

### 5. Exportação para Formato ONNX
Exportação com opset=12 e simplificação do grafo para inferência rápida em CPU via `onnxruntime`.

In [ ]:
caminho_onnx = melhor_modelo.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True
)
print('Modelo ONNX exportado em:', caminho_onnx)

### 6. Verificação do Modelo ONNX com onnxruntime

In [ ]:
import onnxruntime as ort

sessao = ort.InferenceSession('runs/detect/treino_placas/weights/best.onnx', providers=['CPUExecutionProvider'])
print('Entrada:', sessao.get_inputs()[0].name, sessao.get_inputs()[0].shape)
print('Saída:', sessao.get_outputs()[0].name, sessao.get_outputs()[0].shape)
print('Verificação concluída com sucesso!')

### 7. Download do Modelo para Uso no Desktop

In [ ]:
from google.colab import files
# Faz o download do arquivo best.onnx diretamente para sua máquina
files.download('runs/detect/treino_placas/weights/best.onnx')
# Ao baixar, coloque o arquivo na pasta models/placas.onnx do repositório local